# 01 — Data Cleaning and Preparation (2022–2025)

## Objective

This notebook prepares the recent Indian Air Quality Index (AQI) dataset for exploratory data analysis and predictive modelling. The cleaning process standardizes column names and formats, resolves duplicate city-date observations, checks missing values, and organizes the data into a consistent city-wise time series.

The source dataset contains **235,785 observations across 291 cities and 32 states**, covering **1 April 2022 to 30 April 2025**.

The resulting dataset serves as the foundation for the subsequent exploratory analysis, feature engineering, and next-day AQI forecasting experiments.

In [26]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [27]:
raw = pd.read_csv("../data/raw/aqi.csv")
print("Shape:", raw.shape)
raw.head()

Shape: (235785, 9)


,date,state,area,number_of_monitoring_stations,prominent_pollutants,aqi_value,air_quality_status,unit,note
0,30-04-2025,Maharashtra,Amravati,2,PM10,78,Satisfactory,number_of_monitoring_stations in Absolute Numb...,NaN
1,30-04-2025,Bihar,Purnia,1,CO,56,Satisfactory,number_of_monitoring_stations in Absolute Numb...,NaN
2,30-04-2025,Madhya Pradesh,Katni,1,O3,98,Satisfactory,number_of_monitoring_stations in Absolute Numb...,NaN
3,30-04-2025,Chhattisgarh,Tumidih,1,PM10,103,Moderate,number_of_monitoring_stations in Absolute Numb...,NaN
4,30-04-2025,Assam,Byrnihat,1,PM2.5,61,Satisfactory,number_of_monitoring_stations in Absolute Numb...,NaN


In [28]:
print(raw.dtypes)
print()
print("Missing % per column:")
print((raw.isnull().mean() * 100).round(1).sort_values(ascending=False))
print()
print("Unique value count per column:")
print(raw.nunique())


date                                 str
state                                str
area                                 str
number_of_monitoring_stations      int64
prominent_pollutants                 str
aqi_value                          int64
air_quality_status                   str
unit                                 str
note                             float64
dtype: object

Missing % per column:
note                             100.0
date                               0.0
state                              0.0
number_of_monitoring_stations      0.0
area                               0.0
prominent_pollutants               0.0
aqi_value                          0.0
air_quality_status                 0.0
unit                               0.0
dtype: float64

Unique value count per column:
date                             1125
state                              32
area                              291
number_of_monitoring_stations      40
prominent_pollutants               49
aqi_va

In [29]:
print("unit column unique values:", raw['unit'].unique())
print("note column sample (non-null if any):", raw['note'].dropna().unique()[:5])

unit column unique values: <StringArray>
['number_of_monitoring_stations in Absolute Number, aqi_value in Indices']
Length: 1, dtype: str
note column sample (non-null if any): []


## Data Cleaning Methodology

The dataset undergoes the following preprocessing steps:

- **Schema standardization:** Rename key variables to `Date`, `City`, `AQI`, and `AQI_Bucket` for consistency across notebooks.
- **Column selection:** Remove `unit`, which contains constant metadata, and `note`, which contains only missing values.
- **Date conversion:** Convert dates from `DD-MM-YYYY` strings into datetime values.
- **Location standardization:** Normalize city and state names by removing surrounding whitespace and applying consistent capitalization.
- **Duplicate removal:** Retain one observation per city and date.
- **Missing-value checks:** Verify the availability of essential AQI, date, location, and air-quality-category fields.
- **Chronological ordering:** Sort observations by city and date to support subsequent time-series feature engineering.

These steps produce a standardized dataset suitable for further analysis without introducing forecasting targets or lag-based features at the cleaning stage.

In [30]:
df = raw.rename(columns={
    'area': 'City',
    'aqi_value': 'AQI',
    'air_quality_status': 'AQI_Bucket',
    'date': 'Date'
})
df.columns.tolist()

['Date',
 'state',
 'City',
 'number_of_monitoring_stations',
 'prominent_pollutants',
 'AQI',
 'AQI_Bucket',
 'unit',
 'note']

In [31]:
df = df.drop(columns=['unit', 'note'])
df.shape

(235785, 7)

In [32]:
print(df['Date'].head(10).tolist())

['30-04-2025', '30-04-2025', '30-04-2025', '30-04-2025', '30-04-2025', '30-04-2025', '30-04-2025', '30-04-2025', '30-04-2025', '30-04-2025']


In [ ]:

df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce')

print("Unparseable dates:", df['Date'].isnull().sum())
print("Date range:", df['Date'].min(), "to", df['Date'].max())


Unparseable dates: 0
Date range: 2022-04-01 00:00:00 to 2025-04-30 00:00:00


In [34]:
df['City'] = df['City'].astype(str).str.strip().str.title()
df['state'] = df['state'].astype(str).str.strip().str.title()

print("Unique cities:", df['City'].nunique())
print("Unique states:", df['state'].nunique())

Unique cities: 291
Unique states: 32


In [ ]:

sorted(df['City'].unique())[:40]


['Agartala',
 'Agra',
 'Ahmedabad',
 'Ahmednagar',
 'Aizawl',
 'Ajmer',
 'Akola',
 'Alwar',
 'Amaravati',
 'Ambala',
 'Amravati',
 'Amritsar',
 'Anantapur',
 'Angul',
 'Ankleshwar',
 'Araria',
 'Ariyalur',
 'Arrah',
 'Asansol',
 'Aurangabad',
 'Baddi',
 'Badlapur',
 'Bagalkot',
 'Baghpat',
 'Bahadurgarh',
 'Balasore',
 'Ballabgarh',
 'Banswara',
 'Baran',
 'Barbil',
 'Bareilly',
 'Baripada',
 'Barmer',
 'Barrackpore',
 'Bathinda',
 'Begusarai',
 'Belapur',
 'Belgaum',
 'Bengaluru',
 'Bettiah']

In [36]:
dupes = df.duplicated(subset=['City', 'Date']).sum()
print("Duplicate (City, Date) rows found:", dupes)

df = df.drop_duplicates(subset=['City', 'Date'], keep='first')
print("Shape after removing duplicates:", df.shape)

Duplicate (City, Date) rows found: 780
Shape after removing duplicates: (235005, 7)


In [37]:
before = len(df)
df = df.dropna(subset=['AQI', 'AQI_Bucket'])
print(f"Dropped {before - len(df)} rows with missing AQI/AQI_Bucket.")
print(f"Remaining rows: {len(df)}")

Dropped 0 rows with missing AQI/AQI_Bucket.
Remaining rows: 235005


In [38]:
df['AQI_Bucket'] = df['AQI_Bucket'].astype(str).str.strip().str.title()
print(df['AQI_Bucket'].unique())


<StringArray>
['Satisfactory', 'Moderate', 'Good', 'Poor', 'Very Poor', 'Severe']
Length: 6, dtype: str


In [39]:
print(df['AQI_Bucket'].value_counts())
print()
print((df['AQI_Bucket'].value_counts(normalize=True) * 100).round(1))


AQI_Bucket
Satisfactory    88632
Moderate        77166
Good            41913
Poor            21094
Very Poor        5646
Severe            554
Name: count, dtype: int64

AQI_Bucket
Satisfactory    37.7
Moderate        32.8
Good            17.8
Poor             9.0
Very Poor        2.4
Severe           0.2
Name: proportion, dtype: float64


In [40]:
before = len(df)
df = df.dropna(subset=['City', 'Date', 'AQI'], how='any')
print(f"Dropped {before - len(df)} additional incomplete rows.")
print(f"Final shape: {df.shape}")


Dropped 0 additional incomplete rows.
Final shape: (235005, 7)


In [41]:
df = df.sort_values(['City', 'Date']).reset_index(drop=True)
df.head()


,Date,state,City,number_of_monitoring_stations,prominent_pollutants,AQI,AQI_Bucket
0,2022-04-01,Tripura,Agartala,2,"PM10,SO2",78,Satisfactory
1,2022-04-02,Tripura,Agartala,2,"PM10,SO2",77,Satisfactory
2,2022-04-03,Tripura,Agartala,2,"PM2.5,PM10",76,Satisfactory
3,2022-04-04,Tripura,Agartala,2,PM10,57,Satisfactory
4,2022-04-05,Tripura,Agartala,2,PM10,64,Satisfactory


## Cleaning Results and Validation

The cleaning procedure reduced the dataset from **235,785 to 235,005 observations**, removing **780 duplicate city-date records**. No additional rows were removed because of missing essential AQI or category values.

The cleaned dataset retains seven variables: `Date`, `state`, `City`, `number_of_monitoring_stations`, `prominent_pollutants`, `AQI`, and `AQI_Bucket`.

All six air-quality categories are represented. **Satisfactory (37.7%)** and **Moderate (32.8%)** account for the largest proportions of observations, while **Severe (0.2%)** is comparatively rare. This imbalance should be considered when evaluating subsequent classification models.

The cleaned observations are sorted chronologically within each city, establishing a consistent starting point for exploratory analysis and feature construction.

In [43]:
import os
os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/cleaned/aqi_2023_2025_clean.csv", index=False)
print("Saved. Final shape:", df.shape)
df.head()

Saved. Final shape: (235005, 7)


,Date,state,City,number_of_monitoring_stations,prominent_pollutants,AQI,AQI_Bucket
0,2022-04-01,Tripura,Agartala,2,"PM10,SO2",78,Satisfactory
1,2022-04-02,Tripura,Agartala,2,"PM10,SO2",77,Satisfactory
2,2022-04-03,Tripura,Agartala,2,"PM2.5,PM10",76,Satisfactory
3,2022-04-04,Tripura,Agartala,2,PM10,57,Satisfactory
4,2022-04-05,Tripura,Agartala,2,PM10,64,Satisfactory


In [46]:
'''city_day_clean = pd.read_csv("../data/cleaned/city_day_clean.csv", parse_dates=['Date'])

shared_cols = ['City', 'Date', 'AQI', 'AQI_Bucket']

part_1 = city_day_clean[shared_cols].copy()
part_1['source'] = 'city_day_2015_2020'

part_2 = df[shared_cols].copy()
part_2['source'] = 'aqi_2023_2025'

combined_trend = pd.concat([part_1, part_2], ignore_index=True)
combined_trend = combined_trend.sort_values(['City', 'Date']).reset_index(drop=True)

combined_trend.to_csv("../data/processed/aqi_combined_trend.csv", index=False)
print("Combined trend table shape:", combined_trend.shape)
print(combined_trend['source'].value_counts())
combined_trend.head()'''


'city_day_clean = pd.read_csv("../data/cleaned/city_day_clean.csv", parse_dates=[\'Date\'])\n\nshared_cols = [\'City\', \'Date\', \'AQI\', \'AQI_Bucket\']\n\npart_1 = city_day_clean[shared_cols].copy()\npart_1[\'source\'] = \'city_day_2015_2020\'\n\npart_2 = df[shared_cols].copy()\npart_2[\'source\'] = \'aqi_2023_2025\'\n\ncombined_trend = pd.concat([part_1, part_2], ignore_index=True)\ncombined_trend = combined_trend.sort_values([\'City\', \'Date\']).reset_index(drop=True)\n\ncombined_trend.to_csv("../data/processed/aqi_combined_trend.csv", index=False)\nprint("Combined trend table shape:", combined_trend.shape)\nprint(combined_trend[\'source\'].value_counts())\ncombined_trend.head()'

## Summary

This notebook establishes a standardized recent-period AQI dataset for the project. The cleaning process removes duplicate observations, resolves formatting inconsistencies, and preserves the information required for city-wise air-quality analysis.

**Key outcomes:**

- Final dataset: **235,005 observations and 7 variables**.
- Temporal coverage: **April 2022–April 2025**.
- Geographic coverage: **291 cities across 32 states**, as identified during initial inspection.
- Data quality: **780 duplicate city-date records removed**, with no missing essential values requiring further row deletion.
- Class distribution: Air-quality categories are imbalanced, particularly for the less frequent Very Poor and Severe classes.

The cleaned dataset provides the input for **Notebook 02 — Exploratory Data Analysis**, where temporal patterns, geographic differences, pollutant characteristics, and AQI variability are examined.

### Limitations

Removing duplicate city-date records by retaining the first occurrence may discard alternative readings when duplicate records differ. The dataset also contains uneven category representation and does not establish complete daily coverage for every city. These considerations should be accounted for in later analysis and modelling.